# Swings and FVG timing

This deterministic CPU example illustrates a software contract only. It uses no market feed and provides no evidence of trading performance.

In [1]:
import pandas as pd

from features.po3.fvg_detection import FVGConfig, detect_fvg
from features.technical.swings import detect_pivots

idx = pd.date_range("2026-01-05 09:00", periods=8, freq="1min", tz="Etc/GMT-3")
bars = pd.DataFrame(
    {
        "open": [9.0, 10.0, 12.0, 11.0, 9.0, 11.0, 12.0, 12.0],
        "high": [10.0, 12.0, 13.0, 12.0, 11.0, 14.0, 15.0, 13.0],
        "low": [8.0, 11.0, 12.0, 10.0, 8.0, 13.0, 14.0, 12.0],
        "close": [9.0, 11.0, 12.5, 10.5, 9.0, 13.5, 14.5, 12.5],
    },
    index=idx,
)
pivots = detect_pivots(bars, left=1, right=1)
assert pivots.loc[idx[3], "pivot_high_event"]
assert pivots.loc[idx[3], "pivot_high_location"] == 2
assert not pivots.loc[idx[2], "pivot_high_event"]
fvgs = detect_fvg(bars, FVGConfig(min_imbalance_pts=1.0))
assert fvgs.loc[idx[2], "fvg_bullish"] == 1
assert fvgs.loc[idx[4], "fvg_bearish"] == 1
(
    pivots.loc[[idx[2], idx[3]], ["pivot_high_event", "pivot_high_location"]],
    fvgs[["fvg_bullish", "fvg_bearish"]],
)

(                           pivot_high_event  pivot_high_location
 2026-01-05 09:02:00+03:00             False                  NaN
 2026-01-05 09:03:00+03:00              True                  2.0,
                            fvg_bullish  fvg_bearish
 2026-01-05 09:00:00+03:00            0            0
 2026-01-05 09:01:00+03:00            0            0
 2026-01-05 09:02:00+03:00            1            0
 2026-01-05 09:03:00+03:00            0            0
 2026-01-05 09:04:00+03:00            0            1
 2026-01-05 09:05:00+03:00            1            0
 2026-01-05 09:06:00+03:00            1            0
 2026-01-05 09:07:00+03:00            0            0)